# page-blocks-1-3_vs_4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('26_page-blocks-1-3_vs_4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,62.875,56.159,100.000,43.333,100.000,71.667,94.067,99.474,59.517
smote,98.865,85.226,75.159,100.000,97.748,98.874,89.150,99.392,92.046
bsmote,98.447,90.416,85.000,98.333,98.645,98.489,98.135,99.849,94.431
ros,98.753,83.659,72.222,100.000,97.523,98.761,84.586,99.104,91.206
adasyn,86.204,83.182,96.667,75.333,99.775,87.554,96.710,99.732,84.693
casb,98.981,86.081,75.714,100.000,97.972,98.986,91.955,99.523,92.531
casb_relaxed,99.037,86.838,76.905,100.000,98.085,99.042,88.612,99.370,92.938
casb_edited,98.809,84.344,73.393,100.000,97.634,98.817,86.517,99.208,91.576
casb_ros,99.093,87.707,78.810,100.000,98.197,99.099,90.578,99.418,93.400


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,91.893,89.825,97.143,85.333,99.773,92.553,98.381,99.880,90.859
smote,98.392,88.983,82.143,98.333,98.534,98.434,99.524,99.963,93.687
bsmote,99.094,87.596,78.095,100.000,98.197,99.099,100.000,100.000,93.345
ros,99.263,89.794,82.024,100.000,98.534,99.267,100.000,100.000,94.528
adasyn,98.980,86.453,76.429,100.000,97.972,98.986,98.571,99.944,92.716
casb,99.320,90.801,83.810,100.000,98.647,99.323,99.167,99.925,95.060
casb_relaxed,97.578,88.635,83.095,96.667,98.648,97.657,99.524,99.963,93.106
casb_edited,98.863,85.453,75.095,100.000,97.743,98.871,98.135,99.916,92.158
casb_ros,99.207,89.134,80.952,100.000,98.422,99.211,99.167,99.925,94.170


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.060,91.313,96.667,87.000,99.775,93.388,98.889,99.906,92.187
smote,97.017,96.869,100.000,94.333,100.000,97.167,100.000,100.000,96.943
bsmote,97.889,97.778,100.000,96.000,100.000,98.000,100.000,100.000,97.833
ros,98.944,98.889,100.000,98.000,100.000,99.000,100.000,100.000,98.917
adasyn,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000,100.000
casb,97.017,96.869,100.000,94.333,100.000,97.167,100.000,100.000,96.943
casb_relaxed,97.017,96.869,100.000,94.333,100.000,97.167,100.000,100.000,96.943
casb_edited,96.146,95.960,100.000,92.667,100.000,96.333,100.000,100.000,96.053
casb_ros,98.944,98.889,100.000,98.000,100.000,99.000,100.000,100.000,98.917


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,66.126,58.444,90.000,46.667,99.551,73.109,71.962,97.396,62.285
smote,97.371,71.261,55.985,100.000,94.819,97.409,73.353,98.078,84.316
bsmote,97.832,75.378,61.104,100.000,95.718,97.859,71.839,97.821,86.605
ros,93.796,67.774,54.066,93.333,94.819,94.076,74.019,98.116,80.785
adasyn,83.161,65.212,61.079,72.000,96.851,84.426,71.671,97.582,74.186
casb,92.329,68.499,56.187,90.000,95.493,92.746,74.019,98.116,80.414
casb_relaxed,96.635,71.345,56.641,98.333,95.043,96.688,73.353,98.078,83.990
casb_edited,97.258,70.070,54.096,100.000,94.594,97.297,71.515,97.909,83.664
casb_ros,91.297,67.510,55.631,88.000,95.493,91.746,74.219,98.138,79.403


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,78.489,73.935,95.952,65.583,99.775,82.679,90.825,99.164,76.212
smote,97.911,85.585,78.322,98.167,97.775,97.971,90.507,99.358,91.748
bsmote,98.315,87.792,81.050,98.583,98.140,98.362,92.493,99.418,93.054
ros,97.689,85.029,77.078,97.833,97.719,97.776,89.651,99.305,91.359
adasyn,92.086,83.712,83.544,86.833,98.650,92.742,91.738,99.315,87.899
casb,96.912,85.562,78.928,96.083,98.028,97.056,91.285,99.391,91.237
casb_relaxed,97.567,85.922,79.160,97.333,97.944,97.639,90.372,99.352,91.744
casb_edited,97.769,83.957,75.646,98.167,97.493,97.830,89.042,99.258,90.863
casb_ros,97.135,85.810,78.848,96.500,98.028,97.264,90.991,99.370,91.473


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,99.093,smote,98.865,0.228,WIN
1,f1,casb_ros,87.707,bsmote,90.416,-2.709,LOSS
2,precision,casb_ros,78.810,adasyn,96.667,-17.857,LOSS
3,recall,casb,100.000,smote,100.000,0.000,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,99.320,ros,99.263,0.056,WIN
1,f1,casb,90.801,ros,89.794,1.007,WIN
2,precision,casb,83.810,smote,82.143,1.667,WIN
3,recall,casb,100.000,bsmote,100.000,0.000,TIE


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,98.944,adasyn,100.0,-1.056,LOSS
1,f1,casb_ros,98.889,adasyn,100.0,-1.111,LOSS
2,precision,casb,100.000,smote,100.0,0.000,TIE
3,recall,casb_ros,98.000,adasyn,100.0,-2.000,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,97.769,bsmote,98.315,-0.546,LOSS
1,f1,casb_relaxed,85.922,bsmote,87.792,-1.870,LOSS
2,precision,casb_relaxed,79.160,adasyn,83.544,-4.383,LOSS
3,recall,casb_edited,98.167,bsmote,98.583,-0.417,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.408248,0.285714,1.000000,0.166667,1.000000,0.583333,0.910714,0.992509,0.346981,None
1,42,1,svm,smote,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
2,42,1,svm,bsmote,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
3,42,1,svm,ros,1.00,0.988700,0.857143,0.750000,1.000000,0.977528,0.988764,1.000000,1.000000,0.922922,None
4,42,1,svm,adasyn,0.50,0.816497,0.800000,1.000000,0.666667,1.000000,0.833333,1.000000,1.000000,0.808248,None
5,42,1,svm,casb,0.75,0.994366,0.923077,0.857143,1.000000,0.988764,0.994382,1.000000,1.000000,0.958722,None
6,42,1,svm,casb_relaxed,0.50,0.994366,0.923077,0.857143,1.000000,0.988764,0.994382,1.000000,1.000000,0.958722,None
7,42,1,svm,casb_edited,0.50,0.994366,0.923077,0.857143,1.000000,0.988764,0.994382,1.000000,1.000000,0.958722,None
8,42,1,svm,casb_ros,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
9,42,1,knn,original,0.50,0.912871,0.909091,1.000000,0.833333,1.000000,0.916667,0.976190,0.998127,0.910981,None
